In [ ]:
# Run this cell once when starting on notebook.
# On Google Colab, wait for the Google Drive permission prompts before proceeding
import matplotlib.pyplot as plt
import os
import sys
!pip install --quiet datascience
from datascience import *
import numpy as np
import math
DATA_FOLDER="data/"
%matplotlib inline

In [ ]:
# Only run this on Google Colab
# DO NOT RUN on Jupyter Hub !
# It won't do any harm, but it isn't needed and won't work.

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!mkdir -p /content/data
!mkdir -p /content/data/midterm
!gdown --folder https://drive.google.com/drive/folders/1XifjqyxFHvjK6nOUi4doYsG12hGPhOcI?usp=drive_link -O /content/data
!gdown --folder https://drive.google.com/drive/folders/1JsmzNmbTFJ4RYZp2N4L55p2smMNnxcaL?usp=drive_link -O /content/data/midterm
DATA_FOLDER="/content/data/"
!ls -l /content/data


# Data Cleaning, Part 1 (CMPSC 5B, F26)





Real-world datasets are not always *clean*, meaning they may contain some data that's either missing or just plain *wrong*.

The process of taking a data set and removing invalid values is called *data cleaning*.  

It's a process that has to be done thoughtfully, because if not done carefully, it can introduce bias into the data.   But *not* doing it can *also* introduce bias.

In choosing how to clean your data, it's important to know what your purpose is in analyzing the data.

Examples:

* Suppose we are exploring class sizes at UCSB.  If our data set contains courses that are not "regular classroom courses", e.g. individual research classes taught 1-on-1, including those in the data may throw off any kind of averages, medians, etc.
* Suppose we are analyzing salaries for jobs in data science. If our data set contains values that are missing, finding averages, distributions, etc. may result in erroneous conclusions.

Common Data Cleaning Tasks:

* Checking for missing values
* Checking for duplicate entries

Additionally, sometimes datasets are just not in the format that we would prefer:
* They have extra columns we don't need.
* They have column names that are inconvenient to work with.
* They have data in a format other than what we would prefer.

These too, can be considered data cleaning steps.

In the particular case of data that is not precisely in the format that you want it in, you may find it helpful to add new columns computed from existing columns to reformat the data.  We typically use the `apply` function to do this.



## `nan`, `None`, etc.


Some times you'll see values `nan` in datasets, which stands for "Not A Number".  

Sometimes this is a true `nan`, and sometimes it's the string "nan".

Either way, it signifies missing data.

Python value of `None` can also signify missing data.

You need to determine whether your data contains such values, and if so, whether that will impact your analysis, and whether you need to remove the rows that having missing data.



# What's a `nan`?

You'll see lots of references to `nan`.

Let's explore `nan` a bit


In [ ]:
1 + 2 + 3 + np.nan

In [ ]:
np.nan

In [ ]:
1 + np.nan

In [ ]:
1 == np.nan

In [ ]:
np.nan == np.nan # Comparing `nan` with == doesn't work!

In [ ]:
np.nan is np.nan # Comparing `nan` with `is` does work!

In [ ]:
1 is np.nan # This works, but gives us warnings

In [ ]:
np.isnan(1) # But a cleaner way is with the np.isnan(...) function

In [ ]:
np.isnan(np.nan)

In [ ]:
x = np.nan
x == x

In [ ]:
x != x

In [ ]:
np.isnan(x)

## Data Cleaning Example: Data Science Salaries

Our data folder contains a data set of Data Science Salaries that has some missing values in it.  We'll use this as an exercise in data cleaning.

Let's first look at this table:

In [ ]:
salaries_unclean = Table.read_table(DATA_FOLDER + "data_science_salaries_raw.csv")
salaries_unclean

To get to know the data, let's take a few random samples of ten rows.  You may or may not see anything that looks sketchy, but it's a good start.

In [ ]:
salaries_unclean.sample(10)


Next, let's check the Salary column, since that's a column that we might be doing analysis on.  We want to see if any of the values in that column have the special value `nan`, which stands for "Not a Number".

Here's how we can check that:


In [ ]:
# Pull out all of the salaries into a column
salaries = salaries_unclean.column('Salary in USD')
salaries

In [ ]:
len(salaries)

In [ ]:
demo = make_array(1, 2, np.nan, 4, 5)
demo

In [ ]:
np.isnan(demo)

In [ ]:
np.sum(make_array(True, False, False, True, True))

In [ ]:
# Check whether each is the value nan
np.isnan(salaries_unclean.column('Salary in USD'))

In [ ]:
# Count how many of these are true
np.count_nonzero(np.isnan(salaries_unclean.column('Salary in USD')))

So, we could write a function that will take a table and column name, and tell us how many values in that column are the value `nan`:

In [ ]:
def count_nans(table, column_name):
    return np.count_nonzero(np.isnan(table.column(column_name)))

salary_in_USD_nans = count_nans(salaries_unclean, 'Salary in USD')
salary_nans = count_nans(salaries_unclean, 'Salary')
year_nans = count_nans(salaries_unclean, 'Year')
print(f"Number of nans in Salary in USD: {salary_in_USD_nans}")
print(f"Number of nans in Salary: {salary_nans}")
print(f"Number of nans in Year: {year_nans}")

Now this tells us how many `nan` values there are, i.e. whether we need to continue with cleaning.  But it doesn't tell us which rows have those values, and it doesn't remove them from the data set.  So we have more work to do.

Here's a hint:

> Hint: In Python, `nan` is equal to *nothing*, not even itself. So, to check if a variable x is `nan`, you can test x != x.

So, we can't directly check whether a variable is equal to `nan`.  Check this out!



In [ ]:
x = np.nan
x

In [ ]:
type(x)

In [ ]:
type(np.nan)

In [ ]:
np.isnan(x)

In [ ]:
x == x

In [ ]:
x == np.nan

In [ ]:
x != x

In [ ]:
np.isnan(salaries_unclean.column('Salary in USD'))

It appears that testing whether something is a `nan` value can be a bit tricky!  But the `np.isnan` function appears to be a safe way to do it:

In [ ]:
five_rows = salaries_unclean.sample(5)
five_rows

In [ ]:
five_rows.where(make_array(False, True, False, True, True))

In [ ]:
nan_rows = salaries_unclean.where(np.isnan(salaries_unclean.column('Salary in USD')))
nan_rows

Since that worked, let's try the opposite; we'll just put the function `np.logical_not` in front of `np.isnan`:

In [ ]:
valid_rows = salaries_unclean.where(np.logical_not(np.isnan(salaries_unclean.column('Salary in USD'))))
valid_rows

In [ ]:
valid_rows.num_rows

In [ ]:
nan_rows.num_rows

In [ ]:
valid_rows.num_rows + nan_rows.num_rows

In [ ]:
salaries_unclean.num_rows

Let's check whether this data is clean:


In [ ]:
salary_in_USD_nans = count_nans(valid_rows, 'Salary in USD')
salary_nans = count_nans(valid_rows, 'Salary')
year_nans = count_nans(valid_rows, 'Year')
print(f"Number of nans in Salary in USD: {salary_in_USD_nans}")
print(f"Number of nans in Salary: {salary_nans}")
print(f"Number of nans in Year: {year_nans}")

It appears we got rid of all of the `nan` values!  But as a sanity check, let's make sure that we have the number of rows we expect.

How many rows were in the original dataset?

In [ ]:
salaries_unclean.num_rows

In [ ]:
nan_rows.num_rows

In [ ]:
valid_rows.num_rows

In [ ]:
# Does it add up?
salaries_unclean.num_rows == nan_rows.num_rows + valid_rows.num_rows

It appears that this did the trick!  So `valid_rows` is now our cleaned data set, free of any rows that:

* contain `nan` in `Salary` or
* contain `Salary in USD`.

Example of using with_columns:


In [ ]:
t = Table().with_columns({

    'letter': ['a', 'b', 'c', 'z'],

    'count':  [  9,   3,   3,   1],

    'points': [  1,   2,   2,  10],

})
t

In [ ]:
t.rows

# Removing Duplicates

Here is some code to remove duplicates.  It's a little tricky, but we'll look through it a bit at a time, and try to understand how it works.


First, we'll use someting called `.rows`, which we can use to turn a table into a numpy array.

It is explained at this link if you want to read more about it:
* <https://www.data8.org/datascience/_autosummary/datascience.tables.Table.rows.html#datascience.tables.Table.rows>

In [ ]:
# Convert the table to a numpy array
# https://www.data8.org/datascience/_autosummary/datascience.tables.Table.rows.html#datascience.tables.Table.rows
# type(salaries_unclean.rows)
data = np.array(valid_rows.rows)
data


The variable `data` now contains all of the data in our table, but it's not in table form any more.  We've converted it into a `numpy.ndarray`.  The data is all still there, but represented in a different form.

We can't directly do table methods on it, but we can do `numpy` methods on it, and we are going to use that to look for, and remove duplicates.  Then, we'll turn it back to the `table` format that we know and love.

## Using `np.unique`

The `numpy` function `np.unique` allows us to look through a `numpy` array for duplicates.  It gives us back *two* return values, as shown in this code cell:


In [ ]:
experiment_table = Table().with_columns({

    'letter': ['a', 'a', 'c', 'a'],

    'count':  [  9,   9,   3,   9],

    'points': [  1,   1,   2,  1],

})
experiment_table

In [ ]:
experiment_data = np.array(experiment_table.rows)
experiment_data

In [ ]:
ur_experiment_data, counts_experiment_data = np.unique(experiment_data, axis=0, return_counts=True)

In [ ]:

print("\n\n********* unique rows *********")
print(ur_experiment_data)

print("\n\n********* counts *********")
print(counts_experiment_data)


In [ ]:
# finds unique elements of an array and also returns the number of times each unique item appears
# axis to operate on, here axis=0 is important if we want to find unique rows
# (axis=1 would give us unique columns - doesn't make sense in this case)
unique_rows, counts =  np.unique(data, axis=0, return_counts=True)# gives us two arrays


In [ ]:

print("\n\n********* unique rows *********")
print(unique_rows)

print("\n\n********* counts *********")
print(counts)


In [ ]:
# This code finds the rows that had duplicates among all of the unique rows.
# I'll be honest; I didn't understand how this code worked when I first saw it
# But we can use Gemini AI on Google CoLab to explain it

duplicate_rows = unique_rows[counts > 1]
duplicate_rows

# Making sure everything is accounted for

We originally had this many rows in our dataset before we started removing duplicates:


In [ ]:
valid_rows.num_rows

We now have this many rows in the unique_rows (note that we use `len` because this is a numpy array, not yet a table):

In [ ]:
len(unique_rows)

That means we removed seven duplicates.   Let's look at the counts array, and find how many of the counts were greater than 1:

In [ ]:
counts[counts > 1]

That looks like seven elements, i.e. seven rows that had a count greater than 1 that were removed.  Each of then appeared twice (our math would be a bit trickier if any of these numbers were bigger than 2).

In [ ]:
len(counts[counts>1])

In [ ]:
# Do these match?
# Does what we started with == what we ended up with  + what we removed ?
valid_rows.num_rows == len(unique_rows) + len(counts[counts>1])


Yes it does! So all rows are accounted for, and `unique_rows` contains the de-duplicated data.

## Putting the table back together

Now that we have the `unique_rows`, we can put the table back together again.  

First, let's go back to our source of data, which was `valid_rows` and get all of the labels:

In [ ]:
unique_rows

In [ ]:
# Column names
valid_rows.labels

In [ ]:
# Putting together a table with those labels and the unique rows
filtered_unique_salaries = Table(valid_rows.labels).with_rows(unique_rows)
filtered_unique_salaries

Let's apply the same technique as before to verify that there are now no duplicates.


In [ ]:
# finds unique elements of an array and also returns the number of times each unique item appears
# axis to operate on, here axis=0 is important if we want to find unique rows
# (axis=1 would give us unique columns - doesn't make sense in this case)

fus_data = np.array(filtered_unique_salaries.rows)

unique_rows_fs, counts_fs = np.unique(fus_data, axis=0, return_counts=True) # gives us two arrays


In [ ]:

print("\n\n********* unique rows fs *********")
print(unique_rows_fs)

print("\n\n********* counts fs *********")
print(counts_fs)



In [ ]:
duplicate_rows_fs = unique_rows_fs[counts_fs > 1]
duplicate_rows_fs

We see that the duplicate_rows_fs variable is an empty array, so we can now be confident there are no duplicates in our data set.  The data set is clean!

In [ ]:
filtered_unique_salaries

## Some other checks we could perform

There are other checks we could perform when cleaning data:

* We could verify that all values in a field are within a reasonable range, by taking the min and max of a column, and verifying that the values seem reasonable.

* We could convert certain categories/values to upper or lower case - For example, if the data had the "Expert" level with several cases such as "EXpert", "EXPERT", "expert".  We might benefit from converting them all to lower of upper cased versions.  This might involve writing a function, using apply to create a new column, and then dropping and renaming the old column.



## Asking ChatGPT about `unique_rows[counts > 1]`

We asked ChatGPT:

```
When using the datascience library and the numpy library, what does this expression do:
  unique_rows[counts > 1]

```

Here's what we got back:

---

The expression `unique_rows[counts > 1]` filters the unique_rows array to only include rows that have a count greater than 1.

To provide more context, `np.unique` returns the unique rows of a given array along with their counts. The result is returned as two arrays: unique_rows contains the unique rows, and counts contains the corresponding counts for each unique row.

By using the boolean condition `counts > 1`, we create a boolean array where each element represents whether the count for the corresponding row is greater than 1.

By applying this boolean array as an index to `unique_rows`, we only select the rows where the count is greater than 1, effectively filtering out the unique rows and keeping duplicate rows.

Here is an example to demonstrate this:

```
import numpy as np

# Create an array with duplicate rows
arr = np.array([[1, 2], [2, 3], [1, 2], [3, 4], [2, 3]])

# Get the unique rows and their counts
unique_rows, counts = np.unique(arr, axis=0, return_counts=True)

# Filter unique rows to only include rows with count > 1
filtered_rows = unique_rows[counts > 1]

# Display the filtered rows
print(filtered_rows)
```

Output:
```
[[1 2]
 [2 3]]
```
The resulting filtered_rows array contains the rows `[1, 2]` and `[2, 3]`, which are the duplicate rows in the original array.
